In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error

In [2]:
df = sns.load_dataset('mpg')

In [3]:
print("Primeras filas del DataFrame:")
display(df.head(10))

print("Ultimas filas del DataFrame:")
print(df.tail(5))

print("Informacion del DataFrame:")
print(df.info())

Primeras filas del DataFrame:


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino
5,15.0,8,429.0,198.0,4341,10.0,70,usa,ford galaxie 500
6,14.0,8,454.0,220.0,4354,9.0,70,usa,chevrolet impala
7,14.0,8,440.0,215.0,4312,8.5,70,usa,plymouth fury iii
8,14.0,8,455.0,225.0,4425,10.0,70,usa,pontiac catalina
9,15.0,8,390.0,190.0,3850,8.5,70,usa,amc ambassador dpl


Ultimas filas del DataFrame:
      mpg  cylinders  displacement  horsepower  weight  acceleration  \
393  27.0          4         140.0        86.0    2790          15.6   
394  44.0          4          97.0        52.0    2130          24.6   
395  32.0          4         135.0        84.0    2295          11.6   
396  28.0          4         120.0        79.0    2625          18.6   
397  31.0          4         119.0        82.0    2720          19.4   

     model_year  origin             name  
393          82     usa  ford mustang gl  
394          82  europe        vw pickup  
395          82     usa    dodge rampage  
396          82     usa      ford ranger  
397          82     usa       chevy s-10  
Informacion del DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null

In [4]:
print("Estadisticas descriptivas del DataFrame:")
print(df.describe())

Estadisticas descriptivas del DataFrame:
              mpg   cylinders  displacement  horsepower       weight  \
count  398.000000  398.000000    398.000000  392.000000   398.000000   
mean    23.514573    5.454774    193.425879  104.469388  2970.424623   
std      7.815984    1.701004    104.269838   38.491160   846.841774   
min      9.000000    3.000000     68.000000   46.000000  1613.000000   
25%     17.500000    4.000000    104.250000   75.000000  2223.750000   
50%     23.000000    4.000000    148.500000   93.500000  2803.500000   
75%     29.000000    8.000000    262.000000  126.000000  3608.000000   
max     46.600000    8.000000    455.000000  230.000000  5140.000000   

       acceleration  model_year  
count    398.000000  398.000000  
mean      15.568090   76.010050  
std        2.757689    3.697627  
min        8.000000   70.000000  
25%       13.825000   73.000000  
50%       15.500000   76.000000  
75%       17.175000   79.000000  
max       24.800000   82.000000  


In [5]:
print("Tipos originales:")
print(df.dtypes)

Tipos originales:
mpg             float64
cylinders         int64
displacement    float64
horsepower      float64
weight            int64
acceleration    float64
model_year        int64
origin              str
name                str
dtype: object


In [6]:
n_duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {n_duplicados}")

Filas duplicadas: 0


In [7]:
print("Verificacion de datos nulos:")
print(df.isnull().sum())

Verificacion de datos nulos:
mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64


In [8]:
X = df.drop(columns=['mpg', 'name'])
y = df['mpg']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
pipe_num = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
pipe_cat = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore'))


In [11]:
preprocesador = make_column_transformer((pipe_num, make_column_selector(dtype_include=np.number)), (pipe_cat, make_column_selector(dtype_include=object)))

In [12]:
modelo = make_pipeline(preprocesador, LinearRegression()).fit(X_train, y_train)
print(f'\nR2 test: {modelo.score(X_test, y_test):.4f}')


R2 test: 0.8449


**Guardar y cargar modelo**

In [13]:
import joblib

Guardar

In [14]:
joblib.dump(modelo, 'modelo_mpg.joblib')
print('Guardado')

Guardado


Cargar

In [15]:
modelo_cargado = joblib.load('modelo_mpg.joblib')
print(f'R2 original: {modelo.score(X_test, y_test):.4f}')
print(f'R2 cargado: {modelo_cargado.score(X_test, y_test):.4f}')

R2 original: 0.8449
R2 cargado: 0.8449


Prediccion de auto nuevo (nulo)

In [16]:
auto = pd.DataFrame([{'cylinders': 4, 'displacement': 150, 'horsepower': np.nan, 'weight': 2800, 'acceleration': 16.0, 'model_year': 78, 'origin': 'usa'}])
print(f'\nConsumo predicho: {modelo_cargado.predict(auto) [0]:.2f} mpg')


Consumo predicho: 24.89 mpg
